In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size ='5' color='black'>ch15. 데이터베이스 연동</font></b>
# 1절. SQLite 데이터 베이스 연결

- SQLite 데이터베이스 : 
    - 데이터베이스 관리 시스템(DBMS) 서버 설치 없이, 간단한 파일 단위로 SQL을 이용해 데이터에 접근할 수 있는 디스크 기반 데이터베이스 
    - C 언어로 작성된 라이브러리
    - 프로토타입(시제품) 제작에 활용
- 프로젝트단계 : 분석 $\rightarrow$ 설계 $\rightarrow$ 구현 $\rightarrow$ 테스트 $\rightarrow$ 고객 배포 $\rightarrow$ 유지보수
    - 프로토타입 단계: SQLite (빠르고 가벼운 검증용 DB)
    - 완제품/실서버 단계: Oracle, MySQL, MariaDB, PostgreSQL, MSSQL, Amazon AuroraDB 등 (대용량·고성능 처리용 DB)

- [DB Browser for SQLite](https://sqlitebrowser.org/) $\rightarrow$ "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축 풀기

## 1.1 SQLite browser 및 sqlite3 pkg load

In [ ]:
import sqlite3
sqlite3.sqlite_version

In [ ]:
import pandas as pd
pd.__version__

## 1.2 데이터베이스 연결
1. DB 연결 객체 생성 (파일이 없으면 자동 생성, 있으면 연결)
2. 커서 객체 생성
3. 원하는 로직 수행 (SQL 전송 및 결과 처리)
4. 커서 객체 해제
5. DB 연결 객체 해제

In [ ]:
# DB연결 (여기서 에러가 날 경우 VC_redist.x64.exe 설치)
conn = sqlite3.connect('data/ch15_example.db')
conn

In [ ]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결과를 받는 객체
cursor = conn.cursor()
cursor

In [ ]:
cursor.execute('''
    CREATE TABLE MEMBER (
        NAME TEXT,
        AGE  INT,
        EMAIL TEXT 
    )
''')

In [ ]:
cursor.execute('DROP TABLE MEMBER')

In [ ]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount (변경/삭제/삽입된 행의 개수를 정수로 반환)
sql = 'INSERT INTO MEMBER VALUES (\'마길동\', 25, \'h@h.com\')'
cursor.execute(sql) # sql 전송
print('insert, update, delete문의 수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동', 30, 's@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동', 35, 'sil@h.com')"
cursor.execute(sql) 
print('수행 결과 행수 :', cursor.rowcount)

In [ ]:
conn.commit() # commit()과 rollback()은 데이터베이스의 상태를 직접적으로 변경하는 DML 실행 시에만 사용

In [ ]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE") # SELECT sql문 전송

In [ ]:
# SELECT문 실행결과를 받는 함수들
    # fetchone() : 결과에서 한 행만 가져올 때 (데이터가 없으면 None), 튜플 ( )
    # fetchall() : 조회된 모든 행을 한 번에 가져올 때, 튜플 리스트 [( ), ( )]
    # fetchmany(n) : 지정한 n개의 행만 가져올 때, 튜플 리스트 [( ), ( )]
cursor.fetchall()

In [ ]:
cursor.fetchall() # 한 번 읽어서 소모한 데이터는 커서 내에서 다시 접근하거나 재사용할 수 없음

In [ ]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

In [ ]:
# 한 줄씩 가져와 dict list에 append하기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member = cursor.fetchone() # 결과에서 한 행만 가져오기
    if member is None:
        break
    members.append({'name':member[0], 'age':member[1], 'email':member[2]})
members

In [ ]:
class Member:
    'Member 테이블의 내용을 받은 객체 타입'
    def __init__(self, name, age, email):
        self.name = name
        self.age  = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

In [ ]:
dbmember = ('홍길동', 25, 'h@h.com')
m = Member(*dbmember)
print(m)

In [ ]:
# 한 줄씩 가져와 객체 list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

In [ ]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

In [ ]:
cursor.close()
conn.close()

## 1.3 SQL구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우가 있음)
- named(추천)

In [ ]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동','신길동')")
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : qmark 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (?, ?)", (name1, name2))
cursor.fetchall()

In [ ]:
# 파라미터 사용하기 : named 방법 이용
name1 = input('검색할 이름1 :')
name2 = input('검색할 이름2 :')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN ('{name1}', '{name2}')")
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)", {'name1':name1,
                                                                      'name2':name2})
cursor.fetchall()

In [ ]:
# 파라미터 이용하기 : named 방법
name = input('이름 : ')
try:
    age = int(input('나이(숫자) : '))
except:
    print('유효하지 않은 나이를 입력할 경우 초기화 됩니다')
    age = 1
email = input('이메일 : ')
cursor.execute("INSERT INTO MEMBER VALUES (:name, :age, :email)", 
              {'name':name, 'age':age, 'email':email}) # sql 전송
conn.commit()
print('수행 결과 행 수 :', cursor.rowcount)

In [ ]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(~ 11g), pip install oracledb(12 ~)

In [ ]:
import cx_Oracle
cx_Oracle.__version__

In [ ]:
# conn 얻어오는 방법 1 : makedsn 사용

# 1. DSN(Data Source Name) 정보 생성
oracle_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')

# 2. DB 연결 (위치 기반 인자 전달: ID, PW, DSN)
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracle_dsn)
conn = cx_Oracle.connect('scott', 'tiger', oracle_dsn)

# 3. DB 연결 해제
conn.close()

In [ ]:
# conn 얻어오는 방법 2 : Easy Connect 문자열 사용 (에러 발생할 경우 VC_redist.x64.exe 설치)

#conn = cx_Oracle.connect(user='scott', password='tiger', dsn='localhost:1521/xe')
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

In [ ]:
# cursor 객체 생성하고 sql문 전송 & 결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

In [ ]:
for emp in emps:
    print(emp)

In [ ]:
cursor.description

In [ ]:
# 컬럼명 나열
[descipt[0] for descipt in cursor.description]

In [ ]:
import pandas as pd
emp_df = pd.DataFrame(emps, 
                     columns=[descipt[0] for descipt in cursor.description])
emp_df

In [ ]:
# 사용자로부터 검색할 이름을 받아 해당 데이터 출력

sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"

ename = input('검색할 이름 : ').upper()
cursor.execute(sql, {'ename':ename})

emp = cursor.fetchone() # 결과가 있으면 해당 데이터를 튜플로, 결과가 없으면 None으로

if emp:
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp], columns=columns)
    display(df)
else:
    print('해당 이름이 없습니다')

In [ ]:
for col, data in zip(columns, emp):
    print("{} : {}".format(col, data if data is not None else '-'))

In [ ]:
cursor.close()
conn.close()

# 3절. MySQL 연결

| 라이브러리 | 특징 |
|:--|:--|
| **mysql-connector-python** | MySQL 공식 커넥터, python으로 구현 |
| **pyMySQL** | 커뮤니티에서 만든 서드파트 라이브러리(경량), python으로 구현, 널리 쓰임 |

- pip install pyMySQL

In [ ]:
%pip install pyMySQL

In [ ]:
import pymysql

# 1. DB 연결 객체 생성
conn = pymysql.connect(
    host='127.0.0.1',       # DB 서버 주소
    # port=3306,            # 포트 번호 (기본값 3306 생략 가능)
    user='root',            # 계정명
    password='mysql',       # 비밀번호
    database='devdb',       # 사용할 데이터베이스
    charset='utf8mb4',      # 한글 및 이모지 깨짐 방지
    autocommit=True         # DML 자동 커밋 활성화
)

# 2. 커서 객체 생성
cursor = conn.cursor()

# 3. SQL 실행 및 데이터 가져오기
sql = 'SELECT * FROM person'
cursor.execute(sql)         # SQL 전송 및 결과 처리 준비
person = cursor.fetchall()  # 모든 조회 결과 받아오기 (튜플 리스트 형태)

print(person)

# 4. 자원 해제 (생성의 역순)
cursor.close()
conn.close()

In [ ]:
import pandas as pd
import numpy as np
df = pd.DataFrame(person, 
                  columns=[descript[0] for descript in cursor.description])
#df.fillna(np.nan)
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df['hiredate'] = df['hiredate'].astype('datetime64[ns]')

In [ ]:
df = df.astype({'sal':'float64', 'comm':np.float64})
df.info()

# 4절. 연습문제

## oracle연동
- 회원가입 | 전체조회 | 이름찾기 | 메일삭제 | csv내보내기 | 종료

### 0. 처음실행

In [ ]:
def load_conn():
    global conn # 전역변수로 사용함을 선언
    import cx_Oracle
    conn = cx_Oracle.connect("scott", "tiger", "localhost:1521/xe")
load_conn()

### 1. 회원입력

In [ ]:
def fn1_insert_member():
    '사용자로부터 이름, 전화, 이메일, 나이, 등급(1~5)을 입력받아 DB에 insert 한다'
    
    # 사용자로부터 데이터 입력 받기
    name = input('이름 : ')
    phone = input('전화 : ')
    email = input('메일 : ')
    try:
        age = int(input('나이 : '))
        if age<0 :
            age = 0
    except:
        print('유효하지 않은 나이 입력 시 0으로 초기화')
        age = 0
    try:
        grade = int(input('등급(1~5) : '))
        if grade < 1:
            grade = 1
        elif grade > 5:
            grade = 5
    except:
        print('유효하지 않은 등급 입력 시 1로 초기화')
        grade = 1
    
    # SQL문 전송 및 결과 받기
    cursor = conn.cursor()
    sql = "INSERT INTO MEMBER VALUES (:name, :phone, :email, :age, :grade)"
    cursor.execute(sql, {'name':name, 'phone':phone, 'email':email, 'age':age, 'grade':grade})
    
    if cursor.rowcount:
        conn.commit()
        print(name + '님 회원가입 완료')
        
    cursor.close()
    
# fn1_insert_member() 

### 2. 전체조회

In [ ]:
def fn2_display_members():
    'member 테이블의 내용을 데이터프레임으로 display'
    
    import pandas as pd
    
    cursor = conn.cursor()
    
    sql = """SELECT NAME, PHONE, EMAIL, AGE, GRADE 
                FROM MEMBER 
                ORDER BY AGE"""
    cursor.execute(sql)
    
    members = cursor.fetchall() # 데이터가 있으면 튜플 리스트, 데이터가 없으면 빈 리스트
    
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        display(df)
    else:
        print('입력된 회원이 없습니다')
        
    cursor.close()
    
# fn2_display_members()

### 3. 이름으로 조회

In [ ]:
def fn3_search_name():
    '찾고자하는 이름을 입력받아 해당 이름의 데이터를 데이터프레임으로 display'
    
    name1 = input('검색할 이름 : ')
    
    import pandas as pd
    
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM MEMBER WHERE NAME IN :name", {'name': name1})
    members = cursor.fetchall()
    
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        display(df)
    else:
        print('입력된 회원이 없습니다')
        
    cursor.close()
    
# fn3_search_name()

### 4. 메일로 삭제

In [ ]:
def fn4_delete_member():
    '삭제하고자 하는 회원의 메일을 입력받아 회원정보 delete 하기'
    
    email1 = input('삭제할 회원의 메일 : ')
    
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM MEMBER WHERE UPPER(EMAIL)=UPPER(:email)", {'email': email1})
    members = cursor.fetchall()
    
    if members:
        names = [member[0] for member in members]
        print(f'메일이 {email1}인 {names[0]} 회원 외 {len(members)-1}명의 정보를 삭제하겠습니다')
        cursor.execute("DELETE FROM MEMBER WHERE UPPER(EMAIL)=UPPER(:email)", {'email': email1})
        conn.commit()
    else:
        print('데이터베이스에 없는 메일입니다')
        
    cursor.close()

# fn4_delete_member()

### 5. csv 내보내기
- data/ch15_member.csv로 회원정보 내보내기

In [ ]:
def fn5_save_csv():
    import pandas as pd
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM MEMBER")
    members = cursor.fetchall()
    
    if members:
        columns = [descript[0] for descript in cursor.description]
        df = pd.DataFrame(members, columns=columns)
        df.to_csv('data/ch15_member.csv', index=False)
        print('csv 백업 완료')
    else:
        print('입력된 회원이 없습니다')
        
    cursor.close()

# fn5_save_csv()

### 6. 기능들 합하기

In [ ]:
def main():
    while True:
        menu = input("1. 회원 입력 | 2. 전체 조회 | 3. 이름 찾기 | 4. 정보 삭제 | 5. CSV 백업 | 9. 종료")
        if menu=='1':
            fn1_insert_member()
        elif menu=='2':
            fn2_display_members()
        elif menu=='3':
            fn3_search_name()
        elif menu=='4':
            fn4_delete_member()
        elif menu=='5':
            fn5_save_csv()
        elif menu=='9':
            conn.close()
            break
        else:
            print('잘못된 번호 입니다.')

if __name__=='__main__':
    global conn
    import cx_Oracle
    conn = cx_Oracle.connect("scott","tiger","localhost:1521/xe")
#     import sqlite3
#     conn = sqlite3.connect('data/ch15_member.db')

In [ ]:
main()